# Pré-processamento 3 — Maceió

Este notebook transforma os dados de ocorrências em matrizes de séries temporais:
1. Carrega o CSV com índice de célula do grid
2. Calcula a hora do ano para cada ocorrência
3. Para cada tipo de crime, cria uma matriz (célula × hora) por ano
4. Agrega todos os anos e salva um CSV por tipo de crime

**Entrada:** `maceio_with_grid_index.csv`

**Saída:** 5 arquivos `maceio_{tipo_crime}_all_grid.csv`

## 1. Carregamento dos dados com índice de célula

In [6]:
# Carrega o CSV de Maceió com a coluna 'cell' (índice da célula do grid)
# Converte DATA_HORA para datetime

import pandas as pd

df_crimes = pd.read_csv(
    "./output/maceio/maceio_with_grid_index.csv",
    delimiter=";",
    parse_dates=["DATA_HORA"]
)

## 2. Exploração dos dados

In [7]:
display(df_crimes)
df_crimes.info()
years = sorted(df_crimes['DATA_HORA'].dt.year.unique())
display(years)

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell
0,0,-35.698388,-9.658217,street_robbery,2022-01-01 02:00:00,Maceió,2858
1,1,-35.703405,-9.660662,street_robbery,2022-01-01 02:00:00,Maceió,3004
2,2,-35.710142,-9.653007,vehicle_robbery,2022-01-01 02:00:00,Maceió,3151
3,3,-35.713147,-9.668955,street_robbery,2022-01-01 03:00:00,Maceió,3221
4,4,-35.703983,-9.663635,street_robbery,2022-01-01 04:00:00,Maceió,3003
...,...,...,...,...,...,...,...
72707,72707,-35.756315,-9.662037,street_robbery,2014-12-22 12:50:00,Maceió,4098
72708,72708,-35.715706,-9.652401,vehicle_robbery,2014-12-22 19:00:00,Maceió,3224
72709,72709,-35.755908,-9.586733,residential_robbery,2014-12-23 05:30:00,Maceió,4114
72710,72710,-35.738317,-9.624963,street_robbery,2014-12-23 09:30:00,Maceió,3741


<class 'pandas.DataFrame'>
RangeIndex: 72712 entries, 0 to 72711
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Unnamed: 0   72712 non-null  int64         
 1   LONGITUDE    72712 non-null  float64       
 2   LATITUDE     72712 non-null  float64       
 3   ROUBO_GROUP  72712 non-null  str           
 4   DATA_HORA    72712 non-null  datetime64[us]
 5   CIDADE_FATO  72712 non-null  str           
 6   cell         72712 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 3.9 MB


[np.int32(2012),
 np.int32(2013),
 np.int32(2014),
 np.int32(2015),
 np.int32(2016),
 np.int32(2017),
 np.int32(2018),
 np.int32(2019),
 np.int32(2020),
 np.int32(2021),
 np.int32(2022)]

## 3. Tipos de crime e intervalo temporal

In [8]:
# Identifica os tipos de crime e o intervalo de anos nos dados

crimes_types = sorted(df_crimes['ROUBO_GROUP'].unique())
display(crimes_types)

['commercial_robbery',
 'public_transport_robbery',
 'residential_robbery',
 'street_robbery',
 'vehicle_robbery']

## 4. Cálculo da hora do ano

Para cada ocorrência, calcula quantas horas se passaram desde o início do ano. Isso permite organizar os dados em uma grade temporal uniforme.

In [9]:
# Calcula a hora do ano para cada ocorrência
# Exemplo: 1 de janeiro 00:00 = hora 0, 2 de janeiro 00:00 = hora 24
# Utiliza pandarallel para processamento paralelo

import numpy as np
import datetime
from pandarallel import pandarallel
import os

cpu_count = os.cpu_count() or 1
nb_workers = max(1, min(cpu_count, 24))
pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)


def hour_of_year(dt):
    beginning_of_year = datetime.datetime(
        dt["DATA_HORA"].year, 1, 1, tzinfo=dt["DATA_HORA"].tzinfo
    )
    return pd.Series(
        {
            "hour_year": (dt["DATA_HORA"] - beginning_of_year).total_seconds()
            // 3600
        }
    )


df_crimes
display("Translate date of crime to hour of the year")
df_hour = df_crimes.merge(
    df_crimes.parallel_apply(hour_of_year, axis=1), left_index=True, right_index=True
)
display("Initial shape: ", df_hour.shape)
display("Initial shape: ", df_hour)

INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


'Translate date of crime to hour of the year'

'Initial shape: '

(72712, 8)

'Initial shape: '

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell,hour_year
0,0,-35.698388,-9.658217,street_robbery,2022-01-01 02:00:00,Maceió,2858,2.0
1,1,-35.703405,-9.660662,street_robbery,2022-01-01 02:00:00,Maceió,3004,2.0
2,2,-35.710142,-9.653007,vehicle_robbery,2022-01-01 02:00:00,Maceió,3151,2.0
3,3,-35.713147,-9.668955,street_robbery,2022-01-01 03:00:00,Maceió,3221,3.0
4,4,-35.703983,-9.663635,street_robbery,2022-01-01 04:00:00,Maceió,3003,4.0
...,...,...,...,...,...,...,...,...
72707,72707,-35.756315,-9.662037,street_robbery,2014-12-22 12:50:00,Maceió,4098,8532.0
72708,72708,-35.715706,-9.652401,vehicle_robbery,2014-12-22 19:00:00,Maceió,3224,8539.0
72709,72709,-35.755908,-9.586733,residential_robbery,2014-12-23 05:30:00,Maceió,4114,8549.0
72710,72710,-35.738317,-9.624963,street_robbery,2014-12-23 09:30:00,Maceió,3741,8553.0


## 3. Tipos de crime e intervalo temporal

In [10]:
# Identifica os tipos de crime e o intervalo de anos nos dados

GRID_SIZE = 73

for ct in crimes_types:
    df_per_crime = df_hour[df_hour["ROUBO_GROUP"] == ct].copy()
    df_all_years_list = []
    for y in years:
        dfy = df_per_crime[df_per_crime["DATA_HORA"].dt.year == y].copy()

        # count crimes by (cell, DATA_HORA)
        counts = dfy.groupby(["cell", "hour_year"]).size()
        display(counts)

        grid = counts.unstack("hour_year") 


        beginning_of_y1 = datetime.datetime(y, 1, 1)
        beginning_of_y2 = datetime.datetime(y+1, 1, 1)
        num_hours_total = int((beginning_of_y2 - beginning_of_y1).total_seconds() // 3600)
        grid = grid.reindex(columns=range(num_hours_total))
        grid = grid.reindex(index=range(GRID_SIZE**2))
        df_year = grid.copy()
        df_year.columns = list(map(lambda x: str(x) + f"_{str(y)}", df_year.columns.tolist()))
        df_all_years_list.append(df_year)

  
    df_all = pd.concat(df_all_years_list,axis=1)
    df_all.T.to_csv(f"./output/maceio/maceio_{ct}_all_grid.csv")


cell  hour_year
1277  6891.0       1
2520  963.0        1
      1443.0       1
      4035.0       1
      4923.0       1
                  ..
4564  1587.0       1
      1971.0       1
      6171.0       1
4780  579.0        1
      4059.0       1
Length: 365, dtype: int64

cell  hour_year
412   5857.0       1
556   6964.0       1
844   3984.0       1
916   4297.0       1
1784  7493.0       1
                  ..
4781  5395.0       1
4783  5395.0       1
      7285.0       1
4848  7910.0       1
4991  1054.0       1
Length: 812, dtype: int64

cell  hour_year
701   6865.0       1
1783  825.0        1
2000  8414.0       1
2073  107.0        1
2144  6429.0       1
                  ..
4855  1096.0       1
4929  1984.0       1
4991  1717.0       1
5001  2175.0       1
      4118.0       1
Length: 600, dtype: int64

cell  hour_year
701   973.0        1
987   8737.0       1
988   3265.0       1
2000  6927.0       1
2144  7099.0       1
                  ..
4848  4034.0       1
4849  574.0        1
4854  5106.0       1
4929  6625.0       1
4991  522.0        1
Length: 310, dtype: int64

cell  hour_year
267   4316.0       1
988   117.0        1
1784  5611.0       1
2001  2830.0       1
      2974.0       1
                  ..
4781  6228.0       1
4782  8708.0       1
4783  4461.0       1
4928  8566.0       1
5212  7837.0       1
Length: 334, dtype: int64

...

cell  hour_year
988   7991.0       1
1061  96.0         1
      118.0        1
      7463.0       1
1062  4793.0       1
                  ..
5001  5131.0       1
5066  7942.0       1
5141  3777.0       1
      6575.0       1
5214  7845.0       1
Length: 937, dtype: int64

cell  hour_year
628   6578.0       1
988   4795.0       1
1061  1777.0       1
1349  5376.0       1
1494  2374.0       1
                  ..
4991  519.0        1
      7883.0       1
5001  6874.0       1
5213  4317.0       1
5214  4094.0       1
Length: 538, dtype: int64

cell  hour_year
1277  7390.0       1
1784  1156.0       1
      7796.0       1
1928  7961.0       1
1929  7560.0       1
                  ..
5067  697.0        1
5138  1223.0       1
5140  2036.0       1
5211  919.0        1
5212  5464.0       1
Length: 458, dtype: int64

cell  hour_year
339   3890.0       1
628   7753.0       1
1349  4345.0       1
1353  1188.0       1
1784  5638.0       1
                  ..
5211  5040.0       1
5212  8736.0       1
5214  2585.0       1
      5914.0       1
5285  2836.0       1
Length: 343, dtype: int64

cell  hour_year
120   5597.0       1
340   1770.0       1
484   2276.0       1
844   496.0        1
916   356.0        1
                  ..
4994  8274.0       1
5001  3913.0       1
5065  6427.0       1
5213  8490.0       1
5285  2901.0       1
Length: 502, dtype: int64